In [ ]:
# =============================================================================
# CLOUS: Cross-Linguistic Orthographic Uniqueness Score
# Estimating Cross-Lingual Tokenization Fragmentation Risk:
# A Frequency-Based Uniqueness Score for Geographic Entities
# Rivera (2026) — https://github.com/ric-rivera/CLOUS-tokenization-metrics
# =============================================================================
# NOTEBOOK STRUCTURE (mirrors paper section numbering):
#   Cell 1  — Environment setup & dependencies
#   Cell 2  — Clone repository
#   Cell 3  — §3.1 Dataset: CLDR ingestion & data engineering
#   Cell 4  — §3.2 Character frequency profiles (Leipzig corpora)
#   Cell 5  — §3.3 CLOUS formula implementation
#   Cell 6  — §3.4 Baseline metrics (byte length, char count)
#   Cell 7  — §3.4.1 Character bigram surprisal baseline (Leipzig-sourced)
#   Cell 8  — §3.5 Tokenizer benchmark
#   Cell 9  — §3.6 Statistical analysis (Spearman + BH correction)
#   Cell 10 — §4.1 Figure 1: Correlation heatmap
#   Cell 11 — §4.2 Table 4: Baseline comparison
#   Cell 12 — §4.3 Figure 2: Stratified scatter plots
#   Cell 13 — §4.5 Operational diagnostics (ROC-AUC, quartiles, regression)
#   Cell 14 — Execution layer
# =============================================================================


# ===========================================================================
# CELL 1 — Environment setup & dependencies
# ===========================================================================
"""
Run this cell first. Install all required packages.
If running in Google Colab, your HF_TOKEN must be set in Colab Secrets.
"""

!pip install pycountry tiktoken transformers statsmodels matplotlib seaborn scipy scikit-learn

In [ ]:
# ===========================================================================
# CELL 2 — Clone repository
# ===========================================================================
"""
Clone the repository and change into it.
Skip this cell if you already have the repo locally.
"""

!git clone https://github.com/ric-rivera/CLOUS-tokenization-metrics.git
%cd CLOUS-tokenization-metrics

In [ ]:
# ===========================================================================
# CELL 3 — §3.1 Dataset: CLDR ingestion & data engineering
# ===========================================================================
"""
Ingests official country name localizations from the Unicode CLDR repository.
Applies ISO 3166-1 alpha-2 whitelisting via pycountry to exclude macro-regions.
Tracks fallback entries (is_fallback=True) where CLDR lacks a native translation.
All strings are NFC-normalized at ingestion time.

Paper reference: Section 3.1
"""

import os
import json
import unicodedata
import warnings
import requests
import numpy as np
import pandas as pd
import pycountry
import tiktoken
from transformers import AutoTokenizer
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns

# Hugging Face token (required for Llama 3 and Gemma)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
except ImportError:
    pass  # Running locally — set HF_TOKEN as environment variable


def get_valid_iso_alpha2_codes():
    """Returns the official ISO 3166-1 alpha-2 whitelist via pycountry."""
    return {country.alpha_2 for country in pycountry.countries}


def fetch_cldr_territories(locale):
    """Fetches localized country names from the Unicode CLDR GitHub repository."""
    url = (
        f"https://raw.githubusercontent.com/unicode-org/cldr-json/main/"
        f"cldr-json/cldr-localenames-full/main/{locale}/territories.json"
    )
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()['main'][locale]['localeDisplayNames']['territories']
    return None


def normalize_string(text):
    """Applies Unicode NFC normalization to bind diacritics into single codepoints."""
    return unicodedata.normalize('NFC', text) if text else None


def build_robust_localization_matrix(locales):
    """
    Builds the country × language localization matrix.
    Records is_fallback=True for entries where CLDR lacks a native translation
    (English string substituted). These are excluded from primary analysis.
    """
    print("§3.1 Ingesting CLDR data...")
    iso_whitelist = get_valid_iso_alpha2_codes()
    english_raw = fetch_cldr_territories('en')

    master_data = {
        code: {'en': normalize_string(english_raw.get(code))}
        for code in iso_whitelist if english_raw.get(code)
    }

    missing_data_logs = {locale: 0 for locale in locales if locale != 'en'}

    for locale in locales:
        if locale == 'en':
            continue
        localized_raw = fetch_cldr_territories(locale)
        if not localized_raw:
            print(f"  Warning: Failed to retrieve CLDR data for {locale}")
            continue

        for code in master_data.keys():
            name = localized_raw.get(code)
            if name:
                master_data[code][locale] = normalize_string(name)
                master_data[code][f"{locale}_is_fallback"] = False
            else:
                missing_data_logs[locale] += 1
                master_data[code][locale] = master_data[code]['en']
                master_data[code][f"{locale}_is_fallback"] = True

    print("  Missing localization diagnostics (fallback to English):")
    for locale, count in missing_data_logs.items():
        print(f"    [{locale}]: {count} countries")

    df = pd.DataFrame.from_dict(master_data, orient='index')
    df.index.name = 'iso_alpha2'
    print(f"  Matrix built: {len(df)} countries × {len(locales)} languages")
    return df

In [ ]:
# ===========================================================================
# CELL 4 — §3.2 Character frequency profiles (Leipzig corpora)
# ===========================================================================
"""
Builds language-specific character unigram frequency profiles from the
Leipzig Corpora Collection (Goldhahn et al., 2012). Each profile maps
NFC-normalized single characters to their relative frequency (summing to 1.0).

Corpus files must be placed in ./raw_corpora/ as {locale}_letters.txt.
Download 1M-word corpora from: https://wortschatz.uni-leipzig.de/en/download

Paper reference: Section 3.2
"""

REQUIRED_CORPUS_FILES = [
    'en_letters.txt', 'es_letters.txt', 'fr_letters.txt',
    'ru_letters.txt', 'ar_letters.txt', 'hi_letters.txt',
    'zh-Hans_letters.txt', 'ja_letters.txt', 'ta_letters.txt',
    'tr_letters.txt', 'ko_letters.txt'
]

REQUIRED_WORD_FILES = [
    'en_words.txt', 'es_words.txt', 'fr_words.txt',
    'ru_words.txt', 'ar_words.txt', 'hi_words.txt',
    'zh-Hans_words.txt', 'ja_words.txt', 'ta_words.txt',
    'tr_words.txt', 'ko_words.txt'
]


def check_corpus_files(target_dir: str):
    """
    Verifies required Leipzig character frequency files are present.
    Raises FileNotFoundError with download instructions if any are missing.
    """
    missing = [f for f in REQUIRED_CORPUS_FILES
               if not os.path.exists(os.path.join(target_dir, f))]
    if missing:
        print("\n" + "=" * 60)
        print("MISSING CORPUS FILES")
        print("=" * 60)
        print("The following character frequency files are required:")
        for f in missing:
            print(f"  - {f}")
        print("\nDownload 1M-word corpora from:")
        print("  https://wortschatz.uni-leipzig.de/en/download")
        print("\nFor each language, download the word-frequency file,")
        print("run generate_letters_from_words() to produce _letters.txt,")
        print(f"and place it in: {target_dir}/")
        print("=" * 60 + "\n")
        raise FileNotFoundError(
            f"Missing {len(missing)} corpus file(s). See instructions above."
        )
    print(f"  All {len(REQUIRED_CORPUS_FILES)} character frequency files found.")


class FrequencyProfile:
    """
    Manages language-specific character unigram frequency profiles.
    Applies NFC normalization at read time to prevent duplicate-key corruption.
    Caches parsed profiles as JSON to avoid repeated parsing.

    Paper reference: Section 3.2
    """

    def __init__(self, locale: str, epsilon: float = 1e-7,
                 cache_dir: str = "./cache"):
        self.locale = locale
        self.epsilon = epsilon           # OOV penalty floor (reported in paper)
        self.cache_dir = cache_dir
        self.frequency_map = {}
        os.makedirs(self.cache_dir, exist_ok=True)
        self._load_and_normalize_profile()

    def _load_and_normalize_profile(self) -> None:
        self.frequency_map = self._read_leipzig_collection()

    def _read_leipzig_collection(self) -> dict:
        cache_path = os.path.join(
            self.cache_dir, f"{self.locale}_leipzig_freq.json"
        )
        if os.path.exists(cache_path):
            with open(cache_path, 'r', encoding='utf-8') as f:
                return json.load(f)

        raw_filepath = f"./raw_corpora/{self.locale}_letters.txt"
        if not os.path.exists(raw_filepath):
            raise FileNotFoundError(f"Raw corpus missing: {raw_filepath}")

        aggregated_counts = {}
        with open(raw_filepath, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 3 and parts[0].isdigit():
                    raw_char = parts[1].strip() # CRITICAL: Prevents "e " vs "e" mismatches
                    count = int(parts[2])
                    if len(raw_char) != 1:      # Skip digraphs
                        continue
                    # NFC normalization applied at read time (§3.2)
                    normalized_char = unicodedata.normalize('NFC', raw_char)
                    aggregated_counts[normalized_char] = (
                        aggregated_counts.get(normalized_char, 0) + count
                    )

        total_obs = sum(aggregated_counts.values())
        frequency_map = {
            char: count / total_obs
            for char, count in aggregated_counts.items()
        }

        with open(cache_path, 'w', encoding='utf-8') as f:
            json.dump(frequency_map, f, ensure_ascii=False, indent=2)
        return frequency_map

    def get_rarity_coefficient(self, character: str) -> float:
        """Returns relative frequency; epsilon floor for OOV characters."""
        normalized_char = unicodedata.normalize('NFC', character)
        return self.frequency_map.get(normalized_char, self.epsilon)


def generate_letters_from_words(words_filepath: str,
                                 output_letters_filepath: str):
    """
    Utility: extracts character frequencies from a Leipzig word-frequency file.
    Robustly handles both standard (one entry per line) and flattened formats.
    Produces a letters file in the format expected by FrequencyProfile.
    """
    aggregated_counts = {}

    with open(words_filepath, 'r', encoding='utf-8') as f:
        # Read entire file, replace newlines with tabs, and split into a flat stream of tokens
        tokens = f.read().replace('\n', '\t').split('\t')
        tokens = [t for t in tokens if t.strip()] # Remove empty strings

        # Leipzig format is strictly: rank(0), word(1), absolute_freq(2), relative_freq(3)
        # Process in chunks of 4 to safely handle flattened lines
        for i in range(0, len(tokens), 4):
            if i + 3 < len(tokens) and tokens[i].isdigit():
                word = tokens[i+1]
                try:
                    word_freq = int(tokens[i+2])
                    for char in word:
                        normalized = unicodedata.normalize('NFC', char)
                        aggregated_counts[normalized] = (
                            aggregated_counts.get(normalized, 0) + word_freq
                        )
                except ValueError:
                    continue # Skip malformed chunks

    sorted_chars = sorted(
        aggregated_counts.items(), key=lambda x: x[1], reverse=True
    )
    total = sum(aggregated_counts.values())

    with open(output_letters_filepath, 'w', encoding='utf-8') as f:
        for rank, (char, count) in enumerate(sorted_chars, 1):
            rel_freq = count / total
            f.write(f"{rank}\t{char}\t{count}\t{rel_freq:.8f}\n")

    print(f"  Generated {len(sorted_chars)} entries → {output_letters_filepath}")

In [ ]:
# ===========================================================================
# CELL 5 — §3.3 CLOUS formula implementation
# ===========================================================================
"""
Implements Equation 1 from the paper:

    CLOUS = -(1/N) * sum_{i=1}^{N} log2(max(f(c_i), epsilon))

where N is the NFC-normalized string length, f(c_i) is the relative frequency
of character c_i in the target language corpus, and epsilon = 1e-7 is the
configurable OOV penalty floor applied via max() to prevent undefined values.

CLOUS is computed over all character positions including repetitions, reflecting
the sequential nature of subword tokenization (each position is an independent
potential fracture point).

Paper reference: Section 3.3
"""


def compute_clous_score(text: str, profile: FrequencyProfile) -> tuple[float, int]:
    """
    Computes the CLOUS score for a single NFC-normalized string.
    Returns a tuple: (mean per-character surprisal in bits, OOV character count).
    """
    text = unicodedata.normalize('NFC', text)
    if not text:
        return float('nan'), 0

    surprisal_sum = 0.0
    N = len(text)
    oov_count = 0

    for char in text:
        freq = profile.get_rarity_coefficient(char)
        if freq == profile.epsilon:
            oov_count += 1
        # Explicit max() matches the formula as written in the paper
        surprisal_sum += np.log2(max(freq, profile.epsilon))

    return -(surprisal_sum / N), oov_count


def is_score_valid(entity_length: int, oov_count: int, max_oov_threshold: float = 0.2) -> bool:
    if entity_length == 0:
        return False
    return (oov_count / entity_length) <= max_oov_threshold

def compute_clous_dataframe(country_matrix_df: pd.DataFrame,
                             profiles_dict: dict,
                             target_locales: list) -> pd.DataFrame:
    """
    Applies compute_clous_score to every country × language pair.
    Returns a DataFrame indexed by ISO alpha-2 code.
    Filters out entities with >20% OOV characters to prevent garbage-in/garbage-out.
    """
    print("§3.3 Calculating CLOUS scores...")
    clous_results = {}

    for iso_code, row in country_matrix_df.iterrows():
        clous_results[iso_code] = {}
        for locale in target_locales:
            text = row[locale]
            if not text or pd.isna(text):
                clous_results[iso_code][locale] = np.nan
                continue

            # NEW: Unpack the tuple returned by the updated compute_clous_score
            score, oov_count = compute_clous_score(text, profiles_dict[locale])

            # NEW: Check if the score is valid before saving it
            if not is_score_valid(len(text), oov_count, max_oov_threshold=0.2):
                print(f"  Warning: Skipping '{text}' ({locale}) due to high OOV rate ({oov_count}/{len(text)}).")
                clous_results[iso_code][locale] = np.nan
            else:
                clous_results[iso_code][locale] = score

    return pd.DataFrame.from_dict(clous_results, orient='index')

In [ ]:
# ===========================================================================
# CELL 6 — §3.4 Baseline metrics (byte length, char count)
# ===========================================================================
"""
Defines the two length-based baselines used in Table 4:
  - Byte Length (BL): UTF-8 byte length of the NFC-normalized string
  - Character Count (CC): Unicode codepoint count of the NFC-normalized string

These are computed at analysis time from the benchmark_results DataFrame
and are not stored separately, since they are deterministically derived
from the text column.

Paper reference: Section 3.4
"""


def add_length_baselines(df: pd.DataFrame) -> pd.DataFrame:
    """Adds byte_length and char_count columns to a benchmark DataFrame."""
    df = df.copy()
    df['byte_length'] = df['text'].apply(
        lambda x: len(str(x).encode('utf-8'))
    )
    df['char_count'] = df['text'].apply(lambda x: len(str(x)))
    return df

In [ ]:
# ===========================================================================
# CELL 7 — §3.4.1 Character bigram surprisal baseline (Leipzig-sourced)
# ===========================================================================
"""
Implements the character bigram surprisal baseline (BS):

    BS = -(1/(N-1)) * sum_{i=2}^{N} log2 P(c_i | c_{i-1})

Conditional probabilities estimated with Laplace smoothing (alpha=1.0).

IMPORTANT — corpus sourcing:
Bigram profiles are derived from the same Leipzig 1M-word corpora used for
CLOUS unigram profiles (not from the CLDR country name strings themselves),
ensuring methodological consistency. The {locale}_words.txt files in
raw_corpora/ are the source; bigram counts are extracted by iterating
adjacent character pairs in each word, weighted by word frequency.

Paper reference: Section 3.4.1
"""

def build_bigram_profile_from_leipzig(words_filepath: str, alpha: float = 1.0) -> tuple:
    """
    Robustly builds Laplace-smoothed bigram frequency tables from a Leipzig word file.
    Handles both standard (one entry per line) and flattened (multiple entries per line) formats.
    """
    bigram_counts = {}
    unigram_counts = {}

    with open(words_filepath, 'r', encoding='utf-8') as f:
        # Read entire file, replace newlines with tabs, and split into a flat stream of tokens
        tokens = f.read().replace('\n', '\t').split('\t')
        tokens = [t for t in tokens if t.strip()] # Remove empty strings

        # Leipzig format is strictly: rank(0), word(1), absolute_freq(2), relative_freq(3)
        # Process in chunks of 4 to safely handle flattened lines
        for i in range(0, len(tokens), 4):
            if i + 3 < len(tokens) and tokens[i].isdigit():
                word = tokens[i+1]
                try:
                    word_freq = int(tokens[i+2])
                    normalized_word = unicodedata.normalize('NFC', word)

                    # Unigram counts
                    for char in normalized_word:
                        unigram_counts[char] = unigram_counts.get(char, 0) + word_freq

                    # Bigram counts
                    for j in range(len(normalized_word) - 1):
                        bg = (normalized_word[j], normalized_word[j+1])
                        bigram_counts[bg] = bigram_counts.get(bg, 0) + word_freq

                except ValueError:
                    continue # Skip malformed chunks

    return bigram_counts, unigram_counts


def compute_bigram_surprisal(text: str,
                              bigram_counts: dict,
                              unigram_counts: dict,
                              alpha: float = 1.0) -> float:
    """
    Computes mean conditional character surprisal for a string.
    Laplace smoothing: P(c2|c1) = (count(c1,c2)+alpha) / (count(c1)+alpha*V)
    where V is the unigram vocabulary size.
    """
    text = unicodedata.normalize('NFC', str(text))
    if len(text) < 2:
        return 0.0

    vocab_size = len(unigram_counts)
    total = 0.0
    n = 0

    for i in range(len(text) - 1):
        bg = (text[i], text[i + 1])
        prev = text[i]
        num = bigram_counts.get(bg, 0) + alpha
        denom = unigram_counts.get(prev, 0) + alpha * vocab_size
        total += -np.log2(num / denom)
        n += 1

    return total / n if n > 0 else 0.0


def load_bigram_profiles(locales: list,
                          raw_corpora_dir: str = "./raw_corpora",
                          cache_dir: str = "./cache") -> dict:
    """
    Loads or builds bigram profiles for all target locales.
    Uses Leipzig word files ({locale}_words.txt) as the corpus source.
    Caches parsed profiles as JSON to avoid repeated parsing.
    """
    profiles = {}
    os.makedirs(cache_dir, exist_ok=True)

    for locale in locales:
        cache_path = os.path.join(cache_dir, f"{locale}_bigram_profile.json")

        if os.path.exists(cache_path):
            with open(cache_path, 'r', encoding='utf-8') as f:
                cached = json.load(f)
            # JSON keys are strings; restore tuple keys for bigram_counts
            bigram_counts = {
                tuple(k.split('\x00')): v
                for k, v in cached['bigram_counts'].items()
            }
            unigram_counts = cached['unigram_counts']
            profiles[locale] = (bigram_counts, unigram_counts)
            continue

        words_filepath = os.path.join(raw_corpora_dir, f"{locale}_words.txt")
        if not os.path.exists(words_filepath):
            print(
                f"  Warning: {locale}_words.txt not found. "
                f"Bigram profile for {locale} will be empty."
            )
            profiles[locale] = ({}, {})
            continue

        print(f"  Building bigram profile for {locale} from Leipzig corpus...")
        bc, uc = build_bigram_profile_from_leipzig(words_filepath)
        profiles[locale] = (bc, uc)

        # Cache with string keys (JSON cannot serialize tuple keys)
        serializable = {
            'bigram_counts': {
                f"{k[0]}\x00{k[1]}": v for k, v in bc.items()
            },
            'unigram_counts': uc
        }
        with open(cache_path, 'w', encoding='utf-8') as f:
            json.dump(serializable, f, ensure_ascii=False)

    return profiles


def add_bigram_surprisal_column(df: pd.DataFrame,
                                 bigram_profiles: dict) -> pd.DataFrame:
    """Adds bigram_surprisal column to benchmark DataFrame."""
    df = df.copy()

    def _compute(row):
        locale = row['locale']
        if locale not in bigram_profiles:
            return float('nan')
        bc, uc = bigram_profiles[locale]
        if not bc:
            return float('nan')
        return compute_bigram_surprisal(row['text'], bc, uc)

    df['bigram_surprisal'] = df.apply(_compute, axis=1)
    return df

In [ ]:
# ===========================================================================
# CELL 8 — §3.5 Tokenizer benchmark
# ===========================================================================
"""
Computes tokenization fertility for all country × language × tokenizer triples.
Fertility = token_count / character_count (Ács 2019; Rust et al. 2021).
Special tokens disabled (add_special_tokens=False) to isolate string properties.

Tokenizer registry covers three algorithm families:
  BPE:          GPT-4o (o200k_base), GPT-4 (cl100k_base), Llama 3.1-8B
  SentencePiece: Gemma-7B, XLM-RoBERTa
  WordPiece:    mBERT (bert-base-multilingual-cased)

Paper reference: Sections 3.5–3.6
"""

TOKENIZER_REGISTRY = {
    "gpt4o":  {"backend": "tiktoken",      "model": "o200k_base"},
    "gpt4":   {"backend": "tiktoken",      "model": "cl100k_base"},
    "llama3": {"backend": "transformers",  "model": "meta-llama/Meta-Llama-3.1-8B"},
    "gemma":  {"backend": "transformers",  "model": "google/gemma-7b"},
    "xlmr":   {"backend": "transformers",  "model": "xlm-roberta-base"},
    "mbert":  {"backend": "transformers",  "model": "bert-base-multilingual-cased"},
}


def load_tokenizer(name: str):
    config = TOKENIZER_REGISTRY[name]
    if config["backend"] == "tiktoken":
        return tiktoken.get_encoding(config["model"])
    return AutoTokenizer.from_pretrained(config["model"])


def compute_fertility(tokenizer, text: str, backend: str) -> float:
    """
    Fertility = tokens / characters.
    Special tokens excluded for all HuggingFace models.
    """
    if backend == "tiktoken":
        token_count = len(tokenizer.encode(text))
    else:
        token_count = len(tokenizer.encode(text, add_special_tokens=False))
    return token_count / len(text)


def run_tokenization_benchmark(country_matrix_df: pd.DataFrame,
                                clous_scores_df: pd.DataFrame,
                                target_locales: list) -> pd.DataFrame:
    """
    Runs fertility benchmark across all tokenizers × countries × languages.
    Yields ~16,434 rows (249 countries × 11 languages × 6 tokenizers).
    """
    print("§3.5 Running tokenization benchmarks...")
    results = []

    for tokenizer_name, config in TOKENIZER_REGISTRY.items():
        print(f"  -> Testing {tokenizer_name}...")
        try:
            tok = load_tokenizer(tokenizer_name)
        except Exception as e:
            print(f"     Skipping {tokenizer_name}: {e}")
            continue

        for iso_code, row in country_matrix_df.iterrows():
            for locale in target_locales:
                text = row[locale]
                is_fallback = row.get(f"{locale}_is_fallback", False)
                fertility = compute_fertility(tok, text, config["backend"])
                clous = clous_scores_df.loc[iso_code, locale]

                results.append({
                    "iso_alpha2":     iso_code,
                    "locale":         locale,
                    "tokenizer":      tokenizer_name,
                    "text":           text,
                    "is_fallback":    is_fallback,
                    "fertility_score": fertility,
                    "clous_score":    clous,
                })

    return pd.DataFrame(results)

In [ ]:
# ===========================================================================
# CELL 9 — §3.6 Statistical analysis (Spearman + BH correction)
# ===========================================================================
"""
Primary statistic: Spearman rank correlation (ρ), chosen over Pearson because
CLOUS and fertility scores are right-skewed distributions for which normality
cannot be assumed.

Multiple comparisons correction: Benjamini-Hochberg FDR (66 total tests,
one per language × tokenizer combination; 65 valid after mBERT × zh-Hans
exclusion for constant fertility — see Appendix B).

Kendall's τ computed in parallel as a robustness check; directional agreement
reported in Section 4.1.

Paper reference: Section 3.6
"""

# Script family mapping (used in Table 4 and Figure 2)
SCRIPT_MAP = {
    'en': 'Latin',       'es': 'Latin',       'fr': 'Latin',
    'tr': 'Latin+',      'ru': 'Cyrillic',    'ar': 'Arabic',
    'hi': 'Devanagari',  'ta': 'Tamil',       'ko': 'Hangul',
    'zh-Hans': 'Logographic',                  'ja': 'Mixed-script'
}


def analyze_clous_fertility_correlation(results_df: pd.DataFrame) -> pd.DataFrame:
    """
    Computes Spearman ρ and Kendall τ for each locale × tokenizer combination.
    Applies Benjamini-Hochberg correction to all Spearman p-values.
    Excludes mBERT × zh-Hans (constant fertility — see Appendix B).
    """
    print("§3.6 Running statistical correlations...")
    correlation_results = []

    for tokenizer in results_df['tokenizer'].unique():
        for locale in results_df['locale'].unique():
            subset = results_df[
                (results_df['tokenizer'] == tokenizer) &
                (results_df['locale'] == locale) &
                (results_df['is_fallback'] == False)
            ].dropna(subset=['clous_score', 'fertility_score'])

            if len(subset) < 30:
                continue

            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                spearman_r, spearman_p = stats.spearmanr(
                    subset['clous_score'], subset['fertility_score']
                )
                kendall_tau, kendall_p = stats.kendalltau(
                    subset['clous_score'], subset['fertility_score']
                )

            correlation_results.append({
                'tokenizer':   tokenizer,
                'locale':      locale,
                'n':           len(subset),
                'spearman_r':  spearman_r,
                'spearman_p':  spearman_p,
                'kendall_tau': kendall_tau,
                'kendall_p':   kendall_p
            })

    corr_df = pd.DataFrame(correlation_results)

    # Benjamini-Hochberg FDR correction (NaN-safe)
    if not corr_df.empty:
        valid_idx = corr_df['spearman_p'].notna()
        if valid_idx.any():
            _, corrected_p, _, _ = multipletests(
                corr_df.loc[valid_idx, 'spearman_p'], method='fdr_bh'
            )
            corr_df['spearman_p_corrected'] = float('nan')
            corr_df.loc[valid_idx, 'spearman_p_corrected'] = corrected_p
            corr_df['significant_corrected'] = (
                corr_df['spearman_p_corrected'] < 0.05
            )

    sig = corr_df['significant_corrected'].sum()
    valid = corr_df['spearman_r'].notna().sum()
    print(f"  Significant (BH-corrected): {sig} of {valid} valid combinations")
    return corr_df

In [ ]:
# ===========================================================================
# CELL 10 — §4.1 Figure 1: Correlation heatmap
# ===========================================================================
"""
Generates Figure 1: Spearman ρ correlation matrix (11 languages × 6 tokenizers).
Languages grouped by script family with divider between Latin and non-Latin blocks.
Significance marked with ★ (BH-corrected p < 0.05).
mBERT × zh-Hans cell excluded (constant fertility — Appendix B).

Paper reference: Section 4.1, Figure 1
"""


def generate_correlation_heatmap():
    print("§4.1 Generating Figure 1 (Heatmap)...")

    tokenizers = ['Gemma-7B', 'GPT-4', 'GPT-4o', 'Llama 3.1', 'mBERT', 'XLM-R']
    languages = [
        'English (en)', 'Spanish (es)', 'French (fr)', 'Turkish (tr)',
        'Russian (ru)', 'Arabic (ar)', 'Hindi (hi)', 'Tamil (ta)',
        'Korean (ko)', 'Mandarin (zh-Hans)', 'Japanese (ja)'
    ]

    rho_values = [
        [0.413, 0.250, 0.316, 0.229, 0.494, 0.372],
        [0.314, 0.396, 0.365, 0.405, 0.418, 0.429],
        [0.265, 0.320, 0.383, 0.316, 0.306, 0.360],
        [0.177, 0.201, 0.186, 0.136, 0.170, 0.280],
        [0.387, 0.404, 0.432, 0.356, 0.338, 0.407],
        [0.212, 0.066, 0.227, 0.142, 0.038, 0.174],
        [0.165, 0.491, 0.120, 0.103, 0.142, 0.202],
        [0.315, 0.162, 0.282, 0.162, 0.279, 0.247],
        [0.161, 0.624, 0.422, 0.449, 0.174, 0.405],
        [0.311, 0.614, 0.494, 0.465, np.nan, 0.339],
        [0.400, 0.686, 0.437, 0.400, 0.389, 0.319],
    ]

    sig_values = [
        [True,  True,  True,  True,  True,  True],
        [True,  True,  True,  True,  True,  True],
        [True,  True,  True,  True,  True,  True],
        [True,  True,  True,  True,  True,  True],
        [True,  True,  True,  True,  True,  True],
        [True,  False, True,  True,  False, True],
        [True,  True,  False, False, True,  True],
        [True,  True,  True,  True,  True,  True],
        [True,  True,  True,  True,  True,  True],
        [True,  True,  True,  True,  False, True],
        [True,  True,  True,  True,  True,  True],
    ]

    df_rho = pd.DataFrame(rho_values, index=languages, columns=tokenizers)

    annot_data = []
    for i in range(len(languages)):
        row_annots = []
        for j in range(len(tokenizers)):
            val = rho_values[i][j]
            if pd.isna(val):
                row_annots.append("—")
            else:
                star = " ★" if sig_values[i][j] else ""
                row_annots.append(f"{val:.3f}{star}")
        annot_data.append(row_annots)

    annot_df = pd.DataFrame(annot_data, index=languages, columns=tokenizers)

    sns.set_theme(style="white")
    plt.figure(figsize=(10, 8))
    ax = sns.heatmap(
        df_rho, annot=annot_df, fmt="", cmap="Blues",
        cbar_kws={'label': 'Spearman Correlation (ρ)'},
        vmin=0.0, vmax=0.7, linewidths=1, linecolor='white', square=False
    )
    ax.hlines([4], *ax.get_xlim(), colors='black', linewidth=2.5)
    ax.text(-0.8, 2,   'Latin\nScripts',    va='center', ha='right',
            fontsize=12, fontweight='bold', color='#333333')
    ax.text(-0.8, 7.5, 'Non-Latin\nScripts', va='center', ha='right',
            fontsize=12, fontweight='bold', color='#333333')

    plt.title(
        "Figure 1. Spearman ρ Correlation: CLOUS vs. Tokenization Fertility",
        fontsize=14, pad=20, fontweight='bold'
    )
    plt.xlabel("Tokenizer Architecture", fontsize=12, labelpad=10)
    plt.ylabel("", fontsize=12)
    plt.xticks(rotation=45, ha='right', fontsize=11)
    plt.yticks(rotation=0, fontsize=11)
    plt.tight_layout()

    out = "./results/Figure_1_CLOUS_Heatmap.png"
    plt.savefig(out, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved → {out}")

In [ ]:
# ===========================================================================
# CELL 11 — §4.2 Table 4: Baseline comparison
# ===========================================================================
"""
Computes Table 4: median Spearman ρ for CLOUS, Bigram Surprisal, Byte Length,
and Character Count against fertility, grouped by script family.

CLOUS outperforms the bigram baseline in 72.3% of valid cells (47/65),
with the advantage concentrated in logographic and Latin-script families.
Bigram surprisal shows a modest advantage for Arabic, Devanagari, and Tamil,
consistent with richer morphophonological constraints in those scripts.

Paper reference: Section 4.2, Table 4
"""


def compute_table4_baseline_comparison(df: pd.DataFrame) -> pd.DataFrame:
    """
    Produces Table 4: median Spearman ρ per script family across all metrics.
    Requires byte_length, char_count, and bigram_surprisal columns in df.
    """
    print("§4.2 Computing Table 4 (baseline comparison)...")

    results = []
    for (tok, loc), grp in df[df['is_fallback'] == False].groupby(
        ['tokenizer', 'locale']
    ):
        grp = grp.dropna(
            subset=['clous_score', 'fertility_score', 'bigram_surprisal']
        )
        if len(grp) < 30 or grp['fertility_score'].std() == 0:
            continue

        r_c,  _ = stats.spearmanr(grp['clous_score'],      grp['fertility_score'])
        r_b,  _ = stats.spearmanr(grp['bigram_surprisal'], grp['fertility_score'])
        r_bl, _ = stats.spearmanr(grp['byte_length'],      grp['fertility_score'])
        r_cc, _ = stats.spearmanr(grp['char_count'],       grp['fertility_score'])

        results.append({
            'script':      SCRIPT_MAP.get(loc, '?'),
            'rho_clous':   r_c,
            'rho_bigram':  r_b,
            'rho_bl':      r_bl,
            'rho_cc':      r_cc,
            'clous_wins':  r_c > r_b
        })

    result_df = pd.DataFrame(results)
    if result_df.empty:
        print("  Warning: No valid data for Table 4. Please ensure bigram_surprisal calculated correctly.")
        return None
    summary = result_df.groupby('script')[
        ['rho_clous', 'rho_bigram', 'rho_bl', 'rho_cc']
    ].median().round(3)
    summary['clous_adv_over_bigram'] = (
        summary['rho_clous'] - summary['rho_bigram']
    ).round(3)

    wins  = result_df['clous_wins'].sum()
    total = len(result_df)

    print(f"\n  Table 4 - Median Spearman ρ by script family:")
    print(summary.to_string())
    print(f"\n  CLOUS > Bigram Surprisal: {wins}/{total} cells ({wins/total*100:.1f}%)")

    # ADD THE NEW LINES RIGHT HERE:
    mean_adv = (result_df['rho_clous'] - result_df['rho_bigram']).mean()
    print(f"  Mean CLOUS advantage: {mean_adv:+.3f} ρ")

    return summary

In [ ]:
# ===========================================================================
# CELL 12 — §4.3 Figure 2: Stratified scatter plots
# ===========================================================================
"""
Generates Figure 2: CLOUS score vs. fertility across all six tokenizers.
Points colored by script family; regression line per panel.
mBERT × zh-Hans excluded (constant fertility).

Paper reference: Section 4.3, Figure 2
"""


def generate_scatter_plots():
    print("§4.3 Generating Figure 2 (Scatter Plots)...")
    df = pd.read_csv('./cache/benchmark_results.csv')
    df = df[df['is_fallback'] == False].dropna(
        subset=['clous_score', 'fertility_score']
    )
    df['Script Family'] = df['locale'].map(SCRIPT_MAP)
    df = df[~((df['tokenizer'] == 'mbert') & (df['locale'] == 'zh-Hans'))]

    sns.set_theme(style="whitegrid", font_scale=1.1)
    g = sns.FacetGrid(
        df, col="tokenizer", col_wrap=3,
        height=4, aspect=1.2, sharex=False, sharey=True
    )
    g.map_dataframe(
        sns.scatterplot, x="clous_score", y="fertility_score",
        hue="Script Family", palette="tab10", alpha=0.7, edgecolor=None, s=40
    )
    g.map_dataframe(
        sns.regplot, x="clous_score", y="fertility_score",
        scatter=False, color='black',
        line_kws={"linestyle": "--", "linewidth": 2, "alpha": 0.8}
    )

    def annotate_corr(data, **kws):
        if len(data) < 2:
            return
        r, p = stats.spearmanr(data['clous_score'], data['fertility_score'])
        ax = plt.gca()
        p_text = "p < 0.001" if p < 0.001 else f"p = {p:.3f}"
        ax.text(
            0.05, 0.95, f"ρ = {r:.3f}\n{p_text}",
            transform=ax.transAxes, ha="left", va="top",
            fontsize=11, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.9)
        )

    g.map_dataframe(annotate_corr)
    g.set_axis_labels("CLOUS Score (Bits/Char)", "Fertility Score (Tokens/Char)")
    g.set_titles(col_template="{col_name}", size=14, weight='bold')
    g.add_legend(
        title="Script Family", title_fontsize='13', fontsize='11',
        bbox_to_anchor=(1.02, 0.5), loc='center left'
    )
    g.figure.subplots_adjust(top=0.9)
    g.figure.suptitle(
        "Architectural Scaling vs. Orthographic Variance\n"
        "Tokenization Fragmentation by Script Family",
        fontsize=16, fontweight='bold', y=0.98
    )

    out = "./results/Figure_2_Stratified_Scatter.png"
    g.figure.savefig(out, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved → {out}")

In [ ]:
# ===========================================================================
# CELL 13 — §4.5 Operational diagnostics (ROC-AUC, quartiles, regression)
# ===========================================================================
"""
Evaluates CLOUS as an operational screening tool rather than a correlational
measure only (Section 4.5). Three analyses:

1. High-Risk Entity Classification
   Top-quartile fertility = positive class.
   CLOUS threshold = top quartile CLOUS score.
   Reports ROC-AUC, Precision, Recall, F1.

2. Quartile-Based Risk Stratification
   Groups entities into CLOUS quartiles (Q1–Q4).
   Reports mean fertility per quartile and Q1→Q4 increase.

3. Simple Linear Regression
   Fertility = β0 + β1 * CLOUS
   Reports R², MAE, RMSE.

Paper reference: Section 4.5
"""


def run_clous_operational_diagnostics(csv_path: str):
    from sklearn.metrics import (precision_score, recall_score,
                                  f1_score, roc_auc_score,
                                  mean_absolute_error, mean_squared_error,
                                  r2_score)
    from sklearn.linear_model import LinearRegression

    df = pd.read_csv(csv_path)
    clous_col    = 'clous_score'
    fertility_col = 'fertility_score'

    # CRITICAL ADDITION: Drop rows where CLOUS or fertility is NaN.
    # This safely excludes entities that were flagged for high OOV rates.
    df = df.dropna(subset=[clous_col, fertility_col])

    print("§4.5 CLOUS as a High-Risk Entity Screening Tool")
    print("=" * 50)

    # 1. High-Risk Entity Classification
    print("\n1. High-Risk Entity Classification")
    f_thresh = df[fertility_col].quantile(0.75)
    c_thresh = df[clous_col].quantile(0.75)
    df['actual_high_risk']    = (df[fertility_col] >= f_thresh).astype(int)
    df['predicted_high_risk'] = (df[clous_col]     >= c_thresh).astype(int)

    roc_auc   = roc_auc_score(df['actual_high_risk'], df[clous_col])
    precision = precision_score(df['actual_high_risk'], df['predicted_high_risk'])
    recall    = recall_score(df['actual_high_risk'],    df['predicted_high_risk'])
    f1        = f1_score(df['actual_high_risk'],        df['predicted_high_risk'])

    print(f"   ROC-AUC:   {roc_auc:.3f}")
    print(f"   Precision: {precision:.3f}")
    print(f"   Recall:    {recall:.3f}")
    print(f"   F1 Score:  {f1:.3f}")

    # 2. Quartile-Based Risk Stratification
    print("\n2. Quartile-Based Risk Stratification")
    df['clous_quartile'] = pd.qcut(
        df[clous_col], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4']
    )
    q_means = df.groupby('clous_quartile', observed=False)[fertility_col].mean()
    for q in ['Q1', 'Q2', 'Q3', 'Q4']:
        print(f"   {q} Mean Fertility: {q_means[q]:.3f}")
    pct = (q_means['Q4'] - q_means['Q1']) / q_means['Q1'] * 100
    print(f"   Fragmentation Increase (Q1→Q4): {pct:.1f}%")

    # 3. Simple Linear Regression
    print("\n3. Simple Linear Regression: Fertility ~ CLOUS")
    X = df[[clous_col]]
    y = df[fertility_col]
    model = LinearRegression().fit(X, y)
    y_pred = model.predict(X)

    r2   = r2_score(y, y_pred)
    mae  = mean_absolute_error(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))

    print(f"   R²:   {r2:.3f}")
    print(f"   MAE:  {mae:.3f}")
    print(f"   RMSE: {rmse:.3f}")
    print(f"   Eq:   Fertility = {model.intercept_:.3f} + {model.coef_[0]:.3f} * CLOUS")

In [ ]:
# ===========================================================================
# CELL 14 — Execution layer
# ===========================================================================
"""
Full pipeline execution. Run all cells above first, then execute this cell.

Prerequisites:
  - ./raw_corpora/{locale}_letters.txt  (for CLOUS unigram profiles)
  - ./raw_corpora/{locale}_words.txt    (for bigram profiles — Leipzig 1M-word)

Download from: https://wortschatz.uni-leipzig.de/en/download
"""

REQUIRED_CORPUS_FILES_EXE = [
    'en_letters.txt', 'es_letters.txt', 'fr_letters.txt',
    'ru_letters.txt', 'ar_letters.txt', 'hi_letters.txt',
    'zh-Hans_letters.txt', 'ja_letters.txt', 'ta_letters.txt',
    'tr_letters.txt', 'ko_letters.txt'
]

TARGET_BCP47 = [
    'en', 'es', 'fr', 'ru', 'ar', 'hi', 'zh-Hans', 'ja', 'ta', 'tr', 'ko'
]

if __name__ == "__main__":
    # Ensure directories exist
    for d in ["./raw_corpora", "./cache", "./results"]:
        os.makedirs(d, exist_ok=True)

    # Verify corpus files
    missing = [f for f in REQUIRED_CORPUS_FILES_EXE
               if not os.path.exists(f"./raw_corpora/{f}")]
    if missing:
        print("Missing corpus files:", missing)
        print("Download from: https://wortschatz.uni-leipzig.de/en/download")
        raise SystemExit(1)
    print("All corpus files present.\n")

    # §3.1 — Ingest CLDR data
    country_matrix_df = build_robust_localization_matrix(TARGET_BCP47)

    # §3.2 — Load unigram frequency profiles
    print("\n§3.2 Loading FrequencyProfiles...")
    profiles_dict = {loc: FrequencyProfile(loc) for loc in TARGET_BCP47}

    # §3.3 — Compute CLOUS scores
    clous_scores_df = compute_clous_dataframe(
        country_matrix_df, profiles_dict, TARGET_BCP47
    )

    # §3.5 — Run tokenization benchmark
    benchmark_results = run_tokenization_benchmark(
        country_matrix_df, clous_scores_df, TARGET_BCP47
    )

    # §3.4 — Add length baselines
    benchmark_results = add_length_baselines(benchmark_results)

    # §3.4.1 — Add bigram surprisal baseline (Leipzig-sourced)
    print("\n§3.4.1 Building bigram profiles from Leipzig word corpora...")
    bigram_profiles = load_bigram_profiles(TARGET_BCP47)
    benchmark_results = add_bigram_surprisal_column(
        benchmark_results, bigram_profiles
    )

    # §3.6 — Statistical analysis
    final_stats = analyze_clous_fertility_correlation(benchmark_results)

    # Save caches
    benchmark_results.to_csv('./cache/benchmark_results.csv', index=False)
    final_stats.to_csv('./cache/final_stats.csv', index=False)
    clous_scores_df.to_csv('./cache/clous_scores.csv')
    print("\nAll caches saved to ./cache/")

    # §4.1 — Figure 1
    generate_correlation_heatmap()

    # §4.2 — Table 4
    compute_table4_baseline_comparison(benchmark_results)

    # §4.3 — Figure 2
    generate_scatter_plots()

    # §4.5 — Operational diagnostics
    run_clous_operational_diagnostics('./cache/benchmark_results.csv')

    print("\n" + "=" * 60)
    print("Pipeline complete. All outputs saved to ./results/")
    print("=" * 60)